# A tile, not a thread
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/demos/week2/Triton_Tiles_Demo_Colab.ipynb)

**Dataflow Architectures for AI, Week 2, Module 5: Spyre and tile languages**  
**Author:** Dr. Kaoutar El Maghraoui

CUDA asks you to program one thread at a time. Triton asks you to program one **tile**: a block of data that one program instance loads, computes on and stores. The Triton compiler decides how the threads, registers and shared memory inside that program are used. The lecture compares this with KTIR, Spyre's tile IR, which also states where each tile lives and which core owns it.

In this notebook you write two Triton kernels and change their tile size. The first is a vector add, which has almost no reuse. The second is a matrix multiply, where the tile size decides how many times each loaded value is reused.

**What you will do**
1. Write a vector-add kernel and check it against PyTorch.
2. Time it for several block sizes and compare with the T4's memory bandwidth.
3. Write a tiled matrix-multiply kernel.
4. Time it for several tile shapes and relate the result to the arithmetic intensity of a tile.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Triton needs a GPU. The notebook takes about two minutes, most of it compiling kernels.

## 1. Check the environment
Triton is installed with PyTorch on Colab. Each kernel is compiled the first time it runs with a new set of tile sizes, so the first call is slow.

In [ ]:
import torch
import triton
import triton.language as tl

print("PyTorch:", torch.__version__, "| Triton:", triton.__version__)
assert torch.cuda.is_available(), "Triton needs a GPU: set Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
T4_PEAK_BW = 320e9         # bytes per second, published GDDR6 bandwidth
T4_PEAK_FP16 = 65e12       # FLOPs per second, published FP16 tensor-core peak

## 2. A vector-add kernel
Each **program** handles one block of `BLOCK_SIZE` elements. `tl.program_id(0)` says which block this program owns. `tl.arange` builds the offsets of every element in the block at once, so the kernel body works on a whole tile, not on one element. The mask turns off the lanes past the end of the array.

The launch grid is the number of programs: the array length divided by the block size, rounded up.

In [ ]:
@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)                          # which tile this program owns
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n                              # the last tile can run past the end
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    tl.store(out_ptr + offsets, x + y, mask=mask)


def triton_add(x, y, block_size):
    out = torch.empty_like(x)
    n = x.numel()
    grid = (triton.cdiv(n, block_size),)            # one program per tile
    add_kernel[grid](x, y, out, n, BLOCK_SIZE=block_size)
    return out


n = 1 << 24                                         # 16.8 million float32 values, 64 MB per tensor
x = torch.randn(n, device="cuda")
y = torch.randn(n, device="cuda")
print("Matches torch:", torch.equal(triton_add(x, y, 1024), x + y))

## 3. Block size and bandwidth
A vector add reads two values and writes one for each addition: 12 bytes per FLOP in FP32. Its arithmetic intensity is about 0.08 FLOPs per byte, far below the ridge point of any accelerator, so the best it can do is run at memory bandwidth.

The cell times the kernel for block sizes from 32 to 8192 with `triton.testing.do_bench`, which warms up, synchronizes and repeats the measurement for you.

In [ ]:
bytes_moved = 3 * n * 4
add_bandwidth = {}                                  # block size -> bytes per second
print(f"{'BLOCK_SIZE':>10} {'programs':>9} {'time':>9} {'bandwidth':>11} {'of peak':>8}")
for bs in (32, 128, 512, 1024, 4096, 8192):
    ms = triton.testing.do_bench(lambda: triton_add(x, y, bs))
    bw = bytes_moved / (ms / 1e3)
    add_bandwidth[bs] = bw
    print(f"{bs:10d} {triton.cdiv(n, bs):9d} {ms:7.3f} ms {bw / 1e9:8.0f} GB/s {bw / T4_PEAK_BW:7.0%}")
ms = triton.testing.do_bench(lambda: x + y)
print(f"{'torch x+y':>10} {'':9} {ms:7.3f} ms {bytes_moved / (ms / 1e3) / 1e9:8.0f} GB/s")

**Reading the output**
- Small blocks are slow. Each program does very little work, and the GPU spends its time scheduling half a million tiny programs.
- From about 128 elements up, the kernel reaches roughly three quarters of the published bandwidth and matches PyTorch's own add. Larger blocks do not help, because the kernel is limited by memory, not by how the work is divided.
- The tile size is a performance choice, not a correctness one. Every block size gives the same result.

## 4. A tiled matrix multiply
For `C = A @ B`, each program computes one `BLOCK_M` by `BLOCK_N` tile of `C`. It walks along the `K` dimension in steps of `BLOCK_K`: load a tile of `A`, load a tile of `B`, multiply them with `tl.dot`, and add the product to an FP32 accumulator. `tl.dot` maps to the tensor cores where the GPU supports it.

The kernel follows the matmul example in the Triton tutorials, without the extra block ordering that the tutorial adds for cache reuse.

In [ ]:
@triton.jit
def matmul_kernel(a_ptr, b_ptr, c_ptr, M, N, K,
                  stride_am, stride_ak, stride_bk, stride_bn, stride_cm, stride_cn,
                  BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr):
    pid_m = tl.program_id(0)                        # which tile row of C
    pid_n = tl.program_id(1)                        # which tile column of C
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)
    a_ptrs = a_ptr + offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak
    b_ptrs = b_ptr + offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_K)):
        k_left = K - k * BLOCK_K
        a = tl.load(a_ptrs, mask=(offs_m[:, None] < M) & (offs_k[None, :] < k_left), other=0.0)
        b = tl.load(b_ptrs, mask=(offs_k[:, None] < k_left) & (offs_n[None, :] < N), other=0.0)
        acc += tl.dot(a, b)                         # one tile-by-tile product
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    c_ptrs = c_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn
    tl.store(c_ptrs, acc.to(tl.float16), mask=(offs_m[:, None] < M) & (offs_n[None, :] < N))


def triton_matmul(a, b, bm, bn, bk):
    M, K = a.shape
    _, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=torch.float16)
    grid = (triton.cdiv(M, bm), triton.cdiv(N, bn))
    matmul_kernel[grid](a, b, c, M, N, K,
                        a.stride(0), a.stride(1), b.stride(0), b.stride(1), c.stride(0), c.stride(1),
                        BLOCK_M=bm, BLOCK_N=bn, BLOCK_K=bk)
    return c


size = 2048
a = torch.randn(size, size, device="cuda", dtype=torch.float16)
b = torch.randn(size, size, device="cuda", dtype=torch.float16)
c = triton_matmul(a, b, 64, 64, 32)
print("Max difference from torch.matmul:", (c.float() - (a @ b).float()).abs().max().item())

**Reading the output**
- The difference comes from FP16 rounding of the output and a different summation order. The values agree to the precision of FP16.

## 5. Tile shape and reuse
Every value of an `A` tile is used `BLOCK_N` times and every value of a `B` tile `BLOCK_M` times. In one step along `K`, a program loads `(BLOCK_M + BLOCK_N) x BLOCK_K` FP16 values and does `2 x BLOCK_M x BLOCK_N x BLOCK_K` FLOPs. Its arithmetic intensity is:

`tile intensity = BLOCK_M x BLOCK_N / (BLOCK_M + BLOCK_N)` FLOPs per byte

Larger tiles reuse each loaded value more. They also need more registers and shared memory, and they produce fewer programs to spread across the GPU's streaming multiprocessors. The cell times several tile shapes and compares them with `torch.matmul`, which calls cuBLAS.

One caution about this GPU. Current Triton releases target compute capability 8.0 and newer (A100, L4, H100). On the T4, which is compute capability 7.5, `tl.dot` runs at a small fraction of the tensor-core peak. The cell still shows the effect of reuse, but read the absolute numbers with that in mind.

In [ ]:
flops = 2 * size ** 3
configs = [(16, 16, 16), (32, 32, 32), (64, 64, 32), (128, 64, 32), (128, 128, 32)]
tile_tflops = {}                                    # tile shape -> TFLOPS
print(f"{'tile M x N x K':>16} {'programs':>9} {'tile intensity':>15} {'time':>9} {'TFLOPS':>7}")
for bm, bn, bk in configs:
    try:
        ms = triton.testing.do_bench(lambda: triton_matmul(a, b, bm, bn, bk))
        ai = bm * bn / (bm + bn)
        programs = triton.cdiv(size, bm) * triton.cdiv(size, bn)
        tile_tflops[f"{bm}x{bn}x{bk}"] = flops / (ms / 1e3) / 1e12
        print(f"{f'{bm} x {bn} x {bk}':>16} {programs:9d} {ai:11.1f} F/B {ms:7.2f} ms {flops / (ms / 1e3) / 1e12:7.1f}")
    except Exception as err:   # a tile can exceed the GPU's shared memory or registers
        print(f"{f'{bm} x {bn} x {bk}':>16} does not fit on this GPU: {type(err).__name__}")
ms = triton.testing.do_bench(lambda: a @ b)
cublas_tflops = flops / (ms / 1e3) / 1e12
print(f"{'torch.matmul':>16} {'':9} {'':15} {ms:7.2f} ms {flops / (ms / 1e3) / 1e12:7.1f}")
print(f"\nPublished T4 FP16 peak: {T4_PEAK_FP16 / 1e12:.0f} TFLOPS")

**Reading the output**
- Going from 16 by 16 to 32 by 32 tiles doubles the tile intensity and roughly doubles the throughput. That is the reuse effect the formula predicts.
- Past 32 by 32, the tiles get slower on the T4, and the 128-wide tiles are much slower. Each program's FP32 accumulator no longer fits in registers, so values spill to slower memory. On an A100 or H100, `tl.dot` uses the tensor cores and tiles of 128 by 128 and larger are the fast choice. The best tile depends on the hardware.
- cuBLAS is far faster than this kernel on the T4: it uses the tensor cores, hand-tuned tiles and software pipelining. Inductor makes the same call on every GPU: it sends large matmuls to cuBLAS and generates Triton for the operations around them, which Week 3 shows.
- This is the lecture's point about software support. The T4 has tensor cores, but a compiler that does not target them leaves most of the hardware unused.

## Same notebook on other GPUs
The same notebook was run on four NVIDIA GPUs: the T4 from Colab's free tier, the L4, the A100 and the H100. The lecture marks values like these as MEASURED, observed on hardware, and SNAPSHOT, valid for one software release. Read the table for the pattern across GPUs, not for exact numbers.

<!-- gpu-table:start -->
| GPU | vector add, GB/s | tile 16x16x16 | tile 32x32x32 | tile 64x64x32 | tile 128x64x32 | tile 128x128x32 | cuBLAS |
|---|---|---|---|---|---|---|---|
| T4 (cc 7.5) | 262 | 1.1 | 2.1 | 1.0 | 0.1 | 0.1 | 32 |
| L4 (cc 8.9) | 236 | 5.8 | 27.3 | 49.7 | 64.9 | 70.5 | 60 |
| A100-40GB (cc 8.0) | 1286 | 11.5 | 78.5 | 150.1 | 180.9 | 181.2 | 225 |
| H100 (cc 9.0) | 2749 | 19.4 | 127.9 | 283.8 | 353.2 | 431.9 | 591 |

The tile and cuBLAS columns are TFLOPS for the 2048 by 2048 FP16 matmul.

*MEASURED on cloud GPUs on 2026-11-18, PyTorch 2.15.0. SNAPSHOT: these values change with library versions and with the host CPU, and your Colab T4 can differ from the T4 row.*
<!-- gpu-table:end -->

**What the comparison shows**
- On the L4, A100 and H100, throughput rises with the tile size up to 128 by 128. This is the reuse effect from section 5. These GPUs have compute capability 8.0 or later, and `tl.dot` runs on their tensor cores.
- On the T4, the same kernel gets slower past 32 by 32, as section 5 showed. The kernel did not change. The hardware support did.
- On the L4, the largest Triton tile is about as fast as cuBLAS or faster for this shape. On the A100 and H100, cuBLAS is still ahead. A simple tiled kernel can be competitive, and the best tile depends on the GPU.
- The vector add reaches a similar large fraction of each GPU's memory bandwidth. It is memory-bound on every GPU.

## Try this
1. In section 3, use a tensor with 1000 elements. Which block sizes now waste most of their lanes on the mask?
2. In section 5, add `(64, 64, 64)` and `(32, 128, 32)`. Does a non-square tile with the same area perform the same?
3. Change `acc.to(tl.float16)` to keep the output in FP32, and allocate `c` as FP32. How much does the extra output traffic cost?

## Summary
- A Triton program works on a tile. The block size sets how much work each program does and how many programs run.
- For a vector add, any reasonable block size reaches memory bandwidth. Tile size cannot raise arithmetic intensity when there is no reuse.
- For a matmul, the tile intensity is `BLOCK_M x BLOCK_N / (BLOCK_M + BLOCK_N)`. Larger tiles reuse more until registers and shared memory run out, and where that happens depends on the GPU.
- A compiler that does not target the hardware's matrix units leaves them unused, whatever the tile size.
- Triton leaves registers and shared memory to its compiler. A scratchpad tile IR such as KTIR also has to state where each tile lives.

Lecture: Week 2, Module 5, slides on Triton tiles and Triton versus CUDA, and Week 3, Module 2, slide 37 (KTIR by example).